# Datasets Sources
- https://www.kaggle.com/competitions/demand-forecasting-kernels-only/models
- https://www.kaggle.com/competitions/m5-forecasting-accuracy/data


# Model
- LSTM, GRU, RNN comparison

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import copy
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Hyperparameter
DATA_DIR = Path("/content")  # Colab: upload the two CSV files here
BATCH_SIZE = 64
EPOCHS = 15
learning_rate = 1e-3
SEED = 7
LOOKBACK = 28
HIDDEN_SIZE = 32
HOLDOUT = 28

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
print("Device:", device, "| PyTorch:", torch.__version__)

In [ ]:
# Build M5 Time-Series Pipeline
day_cols = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv", nrows=0).columns
day_cols = sorted([c for c in day_cols if c.startswith("d_")], key=lambda c: int(c[2:]))
sales = np.zeros(len(day_cols), dtype=np.float64)
for chunk in pd.read_csv(DATA_DIR / "sales_train_evaluation.csv",
                         usecols=day_cols, dtype=np.int32, chunksize=2000):
    sales += chunk[day_cols].sum(axis=0).to_numpy()

calendar = pd.read_csv(DATA_DIR / "calendar.csv", usecols=["d", "date"])
dates = pd.to_datetime(calendar.set_index("d").loc[day_cols, "date"]).to_numpy()
train_end = len(sales) - 2 * HOLDOUT
val_end = len(sales) - HOLDOUT
for name, start, end in [("Train", 0, train_end),
                          ("Validation", train_end, val_end),
                          ("Test", val_end, len(sales))]:
    print(name, str(dates[start])[:10], "to", str(dates[end - 1])[:10], end - start, "days")

In [ ]:
# Scaling and normalization (fit on Train ONLY)
mu = sales[:train_end].mean()
sigma = max(sales[:train_end].std(), 1e-8)
scaled = ((sales - mu) / sigma).astype(np.float32)
X = np.stack([scaled[t - LOOKBACK:t] for t in range(LOOKBACK, len(sales))])
y = scaled[LOOKBACK:]
target_days = np.arange(LOOKBACK, len(sales))

def make_set(mask):
    return TensorDataset(torch.from_numpy(X[mask, :, None]), torch.from_numpy(y[mask]))

train_set = make_set(target_days < train_end)
val_set = make_set((target_days >= train_end) & (target_days < val_end))
test_set = make_set(target_days >= val_end)

In [ ]:
# Build Dataloader
def make_train_loader():
    return DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED))

val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
inputs, targets = next(iter(val_loader))
print("Input:", tuple(inputs.shape), "Target:", tuple(targets.shape))

In [ ]:
# Build Model
class ForecastRNN(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.recurrent = getattr(nn, kind)(1, HIDDEN_SIZE, batch_first=True)
        self.regressor = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        sequence, _ = self.recurrent(x)
        return self.regressor(sequence[:, -1]).squeeze(-1)

In [ ]:
# Inspect shapes

In [ ]:
# Define loss and optimizer

In [ ]:
# Training and Validating

In [ ]:
# Plot learning curves

In [ ]:
# Evaluation